# Notebook 6: Errors, Troubleshooting & Final Challenge
**Time: 45-60 minutes**

In Notebooks 3–5 you fetched data, shaped it, and saved it — when everything went right. Real APIs don't always cooperate: servers go down, keys expire, names get typo'd, networks drop. The difference between a script that ran once on your machine and one you can rely on at work is what happens when a call *fails*. This notebook teaches you to read errors calmly, handle them in code, and know when (and whom) to ask for help. And then — with every clue collected — you'll close the Data Detective case.

### What you'll learn
- Check the status code on every call and branch on success vs failure
- Recognize the common HTTP errors (404, missing API key, 429, 500) by triggering them on purpose
- Write defensive code with `try`/`except` so one bad response can't crash your whole script
- Combine these into one robust "fetch and report" loop
- Close the Data Detective case

In [ ]:
import requests

---
## 6.1 Always check the status code

This should become a habit. Every time you call an API, check whether it worked *before* using the data. `response.ok` is `True` for any success (2xx) status, so you can branch on it directly.

In [ ]:
# fullText=true asks for an EXACT name match, so a search for "netherlands"
# returns the country itself, not partial matches like "Caribbean Netherlands".
response = requests.get(
    "https://restcountries.com/v3.1/name/netherlands",
    params={"fullText": "true"},
)

if response.ok:  # True for any 2xx (success) status code
    data = response.json()
    print(f"Got data for: {data[0]['name']['common']}")
else:
    print(f"Error: {response.status_code}")
    print(response.text)  # Often contains a helpful error message

**Exercise.** Put the status-check habit into practice — on several lookups at once. Loop over the country names below and build one dict of outcomes. For each name: if the call succeeds, store the country's **common name**; if it fails, store the **status code**.

```python
lookups = ["spain", "narnia"]
```

Required variable:
- `lookup_results` — a **dict** mapping each name in `lookups` to either a common name (string) or a status code (integer)

Hint: this is the demo's `if response.ok:` branch, run inside a `for` loop you write yourself — loops are Notebook 2 territory. Keep the exact-match trick from the demo.

In [ ]:
lookups = ["spain", "narnia"]

lookup_results = {}
for name in lookups:
    response = requests.get(
        f"https://restcountries.com/v3.1/name/{name}",
        params={"fullText": "true"},
    )
    if response.ok:
        lookup_results[name] = response.json()[0]["name"]["common"]
    else:
        lookup_results[name] = response.status_code

print(lookup_results)

---
## 6.2 Common errors and what to do

Status codes come in families: anything starting with **4** (400–499) means the problem is on *your* side of the line — a wrong URL, a missing key, calling too fast. Anything starting with **5** (500–599) means the **server** failed — nothing you sent caused it, and nothing you change will fix it.

Let's trigger the two most common ones on purpose, so you recognize them when they happen for real.

In [ ]:
# 404: Not Found — the URL path doesn't lead anywhere ("atlantis" isn't a country)
response = requests.get("https://restcountries.com/v3.1/name/atlantis")
print(f"Status: {response.status_code}")
print(f"Message: {response.text[:200]}")

### Missing API key (401 / 403)

In Notebook 4 you called NASA's picture-of-the-day API with an `api_key`. Here's the same endpoint with the key left out — a real authentication refusal, and the response message tells you exactly what's missing. NASA answers with **403 Forbidden**; many other APIs use **401 Unauthorized** for the same mistake. Either way the fix is the same: check your key.

In [ ]:
# Same NASA endpoint as Notebook 4 — but without the api_key parameter
response = requests.get("https://api.nasa.gov/planetary/apod")
print(f"Status: {response.status_code}")
print(f"Message: {response.text[:200]}")

### Quick reference for status codes

| Code | Meaning | What to do |
|------|---------|------------|
| **401 Unauthorized** | Missing or invalid API key | Check your API key. Contact the API owner for new credentials. |
| **403 Forbidden** | You don't have permission | Contact the API owner, check your access level. |
| **404 Not Found** | URL is wrong or resource doesn't exist | Double-check the URL against the documentation. |
| **429 Too Many Requests** | You're calling too fast | Slow down — add delays between requests (see below). |
| **500 Internal Server Error** | The server itself failed | Not your fault. Wait and retry; if it *keeps* happening, contact the team that owns the API. |
| **502 / 503 / 504** | Other server-side failures (bad gateway, unavailable, timeout) | Same family as 500 — retry later, escalate if it persists. |

### Rate limiting (429)

APIs protect themselves against being called too often. Cross the limit and you get **429 Too Many Requests** until you slow down. We can't demo a real 429 here — that would mean hammering a public API on purpose, and REST Countries happily serves a polite loop like the one below without complaint.

What you *should* know is the remedy. When an API's documentation states a limit (NASA's `DEMO_KEY` allows 30 requests per hour, for example), space out your calls with `time.sleep()`:

In [ ]:
import time

countries_to_fetch = ["netherlands", "belgium", "germany"]

for name in countries_to_fetch:
    response = requests.get(
        f"https://restcountries.com/v3.1/name/{name}",
        params={"fullText": "true"},
    )
    if response.ok:
        print(f"Got: {response.json()[0]['name']['common']}")
    time.sleep(0.5)  # half a second of breathing room between calls

---
## 6.3 Defensive coding with try/except

So far you've *checked* for errors with `if response.ok:`. But some failures never produce a response to check — the network drops, the server doesn't answer, the URL's host doesn't exist. Those failures **crash** your program: Python stops at the failing line and everything after it never runs.

First, see what that looks like. The cell below calls a host that doesn't exist — expect an angry red error, that's the point:

In [ ]:
# This host doesn't exist, so there's no response at all — not even an error code.
# Run it and look at the result: a long red error report, and the print never happens.
response = requests.get("https://restcountries.invalid/v3.1/all", timeout=10)
print("You will never see this line printed.")

That red wall of text is a **traceback** — Python's report of where it gave up. In a notebook it's an annoyance; in a script processing 200 URLs, one bad URL kills the other 199.

`try`/`except` is a new Python statement that lets you plan for this. In plain words: *"**try** to run these lines; if any of them fails, don't crash — jump to the **except** block instead, then carry on."* That's what "handling an error gracefully" means: the program notices the problem, reports it in a way you control, and keeps going.

Here's the same doomed call, wrapped in the minimal form — one `try`, one `except`:

In [ ]:
try:
    response = requests.get("https://restcountries.invalid/v3.1/all", timeout=10)
    print("Got a response!")
except requests.exceptions.RequestException as e:
    print(f"The call failed, but we're still running. The problem: {type(e).__name__}")

print("This line DOES run now — no crash.")

Two new bits of syntax in there:

- `requests.exceptions.RequestException` is the *family name* for everything that can go wrong inside `requests` — connection failures, timeouts, bad statuses. Catching the family catches them all.
- `as e` stores the error in a variable so you can inspect it — `type(e).__name__` gives the error's class name, like `ConnectionError`.

One gap remains: a 404 doesn't crash anything — `requests` hands you the response and leaves the checking to you. `response.raise_for_status()` closes that gap: it does nothing on a success status, and *raises* (deliberately triggers) an exception on any 4xx/5xx. With it, one `except` block handles bad statuses, timeouts, and dead hosts all the same way:

In [ ]:
url = "https://restcountries.com/v3.1/name/netherlands"

try:
    response = requests.get(url, params={"fullText": "true"}, timeout=10)
    response.raise_for_status()  # turn a 4xx/5xx status into an exception
    data = response.json()
    print(f"Got data for {data[0]['name']['common']}")
except requests.exceptions.RequestException as e:
    print(f"Something went wrong: {type(e).__name__}: {e}")

Two finishing touches worth knowing:

- `timeout=10` means "don't wait forever — give up after 10 seconds." Without it, a hanging server can stall your script indefinitely. The give-up raises a `Timeout`, which the family catch above already handles.
- You can write *multiple* `except` blocks to react differently per error type — Python uses the first one that matches:

```python
except requests.exceptions.Timeout:
    print("Timed out — try again later")
except requests.exceptions.HTTPError as e:
    print(f"Bad status: {e}")
```

**Exercise.** Now make `try`/`except` do the work. Call a URL that returns a **404** and use `response.raise_for_status()` so the bad status becomes an exception you can catch.

Required variables:
- `safe_outcome` — a **string**: `"success"` if the call worked, `"failed"` if it raised
- `error_type` — a **string**: the *name* of the exception that was raised

Hint: §6.1's `narnia` lookup is a handy source of 404s, and the syntax for getting an exception's type name is in the demos above.

In [ ]:
url = "https://restcountries.com/v3.1/name/narnia"

try:
    response = requests.get(url, params={"fullText": "true"}, timeout=10)
    response.raise_for_status()
    safe_outcome = "success"
except requests.exceptions.HTTPError as e:
    safe_outcome = "failed"
    error_type = type(e).__name__

print(f"safe_outcome = {safe_outcome}")
print(f"error_type   = {error_type}")

---
## 6.4 JSON parsing errors

One more thing that can go wrong: the response arrives fine (status 200!) but the body isn't JSON — a proxy returned an HTML error page, or you accidentally called a website instead of an API. Then `.json()` itself raises. Let's trigger that for real: the REST Countries *homepage* is a normal HTML page, not an API endpoint.

In [ ]:
response = requests.get("https://restcountries.com")  # the website, not the API
print(f"Status: {response.status_code} — looks fine...")

try:
    data = response.json()
except requests.exceptions.JSONDecodeError:
    print("...but the body isn't JSON!")
    print(f"What we actually got: {response.text[:120]}")

The takeaway: **a 200 doesn't guarantee JSON.** When `.json()` fails, print `response.text` to see what actually came back — usually the fastest clue about what went wrong.

---
## 6.5 Debugging checklist

The next three sections are **reference material** — nothing to code, no exercises. Skim them now, and come back whenever something breaks for real.

When something doesn't work, go through this list:

1. **Check the status code** — Did the request even succeed?
2. **Print the raw response** — `print(response.text[:1000])`
3. **Check your URL** — Copy-paste from documentation
4. **Check your parameters** — Print them before sending
5. **Check authentication** — Is the key correct? In the right place?
6. **Try in browser** — For GET requests, paste the URL in your browser
7. **Check the documentation** — Maybe the API changed
8. **Search the error message** — Someone else probably hit this

---
## 6.6 Network issues

### Proxy errors (corporate networks)

If you're behind a corporate proxy:
```python
proxies = {
    "http": "http://your-proxy:8080",
    "https": "http://your-proxy:8080"
}
response = requests.get(url, proxies=proxies)
```
Ask IT for the correct proxy settings.

### SSL/Certificate errors

Often caused by corporate firewalls inspecting traffic. Talk to IT.

### VPN required

Some internal APIs only work on VPN. If an API suddenly stops working, check your VPN connection.

---
## 6.7 When to ask for help

Escalate when:
- You need access or permissions you don't have
- The API *consistently* returns 500 errors — that's a server-side problem; report it to the team that owns the API
- Network/proxy issues you can't resolve
- You've spent more than 30 minutes on the same error

Know who to contact:
- **API access/keys, or an API that keeps erroring:** the team that owns the API
- **Network/VPN/proxy:** IT support
- **Data questions:** Data team

---
## 6.8 Putting it all together: one robust fetch

Real scripts call many URLs, and any of them can fail differently — a typo gives a 404, a server hiccup gives a 500, a wrong host never connects at all. A robust loop handles whatever happens and records *something useful* for every URL instead of crashing on the first failure.

**Exercise.** Loop over the three URLs below and build one report. For each label:
- if the request succeeds, store the country's **common name**
- if anything goes wrong, store a string that **starts with** `"ERROR"` (include something useful after it, like the error's type name)

```python
urls = {
    "good":       "https://restcountries.com/v3.1/name/iceland",
    "missing":    "https://restcountries.com/v3.1/name/zzzznotacountry",
    "wrong_host": "https://restcountries.invalid/v3.1/all",
}
```

Required variable:
- `fetch_report` — a **dict** with those same three keys, each mapping to a string

Hint: one pattern from this notebook handles every failure here — both the 404 and the dead host — with a single `except`.

In [ ]:
urls = {
    "good":       "https://restcountries.com/v3.1/name/iceland",
    "missing":    "https://restcountries.com/v3.1/name/zzzznotacountry",
    "wrong_host": "https://restcountries.invalid/v3.1/all",
}

fetch_report = {}

for label, url in urls.items():
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        fetch_report[label] = data[0]["name"]["common"]
    except requests.exceptions.RequestException as e:
        fetch_report[label] = f"ERROR: {type(e).__name__}"

for label, outcome in fetch_report.items():
    print(f"{label}: {outcome}")

In [ ]:
# Run this cell to check your error-handling exercises.

required_vars = ["lookup_results", "safe_outcome", "error_type", "fetch_report"]

missing = [v for v in required_vars if v not in globals() or globals()[v] is None]
assert not missing, f"Missing or unfinished: {missing}. Go back and complete those exercises."

# 6.1 - status-check branching
assert isinstance(lookup_results, dict), f"lookup_results should be a dict, got {type(lookup_results).__name__}"
assert set(lookup_results) == {"spain", "narnia"}, (
    f"lookup_results should have exactly the keys 'spain' and 'narnia', got {sorted(lookup_results)}"
)
assert lookup_results["spain"] == "Spain", (
    f"lookup_results['spain'] should be the common name 'Spain', got {lookup_results['spain']!r}"
)
assert lookup_results["narnia"] == 404, (
    f"lookup_results['narnia'] should be the status code 404, got {lookup_results['narnia']!r}"
)

# 6.3 - try/except + raise_for_status
assert safe_outcome == "failed", f"safe_outcome should be 'failed' (the 404 must raise), got {safe_outcome!r}"
assert error_type == "HTTPError", f"error_type should be 'HTTPError', got {error_type!r}"

# 6.8 - robust fetch report
assert isinstance(fetch_report, dict), f"fetch_report should be a dict, got {type(fetch_report).__name__}"
assert set(fetch_report) == {"good", "missing", "wrong_host"}, (
    f"fetch_report should have keys good/missing/wrong_host, got {sorted(fetch_report)}"
)
assert fetch_report["good"] == "Iceland", (
    f"fetch_report['good'] should be 'Iceland', got {fetch_report['good']!r}"
)
assert fetch_report["missing"].startswith("ERROR"), (
    f"fetch_report['missing'] should start with 'ERROR' (the 404), got {fetch_report['missing']!r}"
)
assert fetch_report["wrong_host"].startswith("ERROR"), (
    f"fetch_report['wrong_host'] should start with 'ERROR' (no connection), got {fetch_report['wrong_host']!r}"
)

notebook6_ready_for_clue = True
print("All checks passed — the case file is ready to close.")

---
---
# Data Detective — Closing the Case

You've collected one clue in every notebook. Time to put the evidence together and close the case.

First make sure the checkpoint above passed — it unlocks the case file. Then fill in the five clues you wrote down along the way, and sign off as the lead detective.

In [ ]:
# Enter the clues you collected (replace each None), and sign the case file:

clue_1 = 52.37        # Notebook 1: Amsterdam's latitude
clue_2 = 4.89         # Notebook 2: Amsterdam's longitude
clue_3 = 18           # Notebook 3: Netherlands population in whole millions
clue_4 = 741_657_922  # Notebook 4: total population of Europe (a big number!)
clue_5 = "Italy"      # Notebook 5: 5th largest European country by population (a string)

detective_name = "Detective J. Doe"  # Lead detective on this case — that's you

In [ ]:
# Run this cell to verify your clues and close the case.

import requests
from datetime import date

assert globals().get("notebook6_ready_for_clue", False), (
    "Pass the checkpoint above first — it unlocks the case file."
)

clue_names = ["clue_1", "clue_2", "clue_3", "clue_4", "clue_5", "detective_name"]
unfilled = [name for name in clue_names if globals().get(name) is None]

if unfilled:
    print(f"Still missing: {', '.join(unfilled)}.")
    print("Fill them in above and run this cell again.")
else:
    errors = []
    if round(float(clue_1), 2) != 52.37:
        errors.append("Clue 1 (Amsterdam latitude) is incorrect — check Notebook 1")
    if round(float(clue_2), 2) != 4.89:
        errors.append("Clue 2 (Amsterdam longitude) is incorrect — check Notebook 2")
    if not (17 <= clue_3 <= 18):
        errors.append("Clue 3 (Netherlands population in millions) is incorrect — check Notebook 3")
    if not (700_000_000 <= clue_4 <= 800_000_000):
        errors.append("Clue 4 (total population of Europe) is incorrect — check Notebook 4")
    if clue_5 != "Italy":
        errors.append("Clue 5 (5th largest European country) is incorrect — check Notebook 5")

    if errors:
        print("Not all clues check out:\n")
        for e in errors:
            print(f"  - {e}")
        print("\nGo back and redo the challenges you missed!")
    else:
        # All clues correct! Fetch live Amsterdam weather using the coordinate clues —
        # wrapped in try/except, exactly the defensive pattern from this notebook.
        try:
            weather_response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={"latitude": clue_1, "longitude": clue_2, "current_weather": True},
                timeout=10,
            )
            weather_response.raise_for_status()
            temp = weather_response.json()["current_weather"]["temperature"]
            weather_line = f"Weather at the scene right now: {temp}°C"
        except requests.exceptions.RequestException:
            weather_line = "Weather at the scene: unavailable (the API is down — handled gracefully!)"

    if not errors:
        today = date.today().strftime("%B %d, %Y")

        print()
        print("=" * 58)
        print()
        print("        CASE FILE: EUROPE — STATUS: CLOSED")
        print()
        print(f"   Lead detective: {detective_name}")
        print()
        print("   Evidence on record:")
        print(f"   - Amsterdam located at {clue_1}°N, {clue_2}°E")
        print(f"   - Netherlands population: ~{clue_3} million")
        print(f"   - Europe total population: {clue_4:,}")
        print(f"   - 5th largest country by population: {clue_5}")
        print(f"   - {weather_line}")
        print()
        print(f"   Case closed: {today}")
        print()
        print("=" * 58)
        print()
        print("Nice work, detective. You can now:")
        print("  - Call any REST API with Python")
        print("  - Process and transform JSON data")
        print("  - Save results as CSV or Excel")
        print("  - Handle errors and debug issues")

---

## What's next?

You now have all the skills to independently fetch data from APIs, transform it, and save it. Here are some ideas for practice:

1. **Discover a company API** — ask your team which APIs they use, open its Swagger page (Appendix A shows you how to read one), and explore the structure of its responses
2. **Try a new public API** from the cheat sheet (PokeAPI, Open-Meteo, JSONPlaceholder)
3. **Combine multiple APIs** — e.g., get country data AND weather for each capital

Remember: the pattern is always the same:
1. `requests.get(url, params=...)` — call the API
2. `response.json()` — get the data
3. Loop + `.append()` — process it
4. `pd.DataFrame(rows)` — make a table
5. `df.to_csv()` — save it